# Diabetic 数据集 Governance 实验

本 notebook 用于运行第二篇 governance 框架在 Diabetic 数据集上的实验。

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("当前项目目录：", PROJECT_ROOT)


当前项目目录： e:\yan\组\三支决策\机器学习\BT_TWD_v2


In [ ]:
CONFIG_PATH = "configs/diabetic_bttwd.yaml"

# 是否运行消融实验：
# True：运行 no_cp 和 no_progressive 两个消融；False：只运行 full，跳过消融单元。
RUN_ABLATION = False

print("当前配置文件：", CONFIG_PATH)
print("运行消融实验：", RUN_ABLATION)


当前配置文件： configs/diabetic_bttwd.yaml
运行消融实验： True


## 结果摘要工具函数


In [3]:
import pandas as pd
from pathlib import Path
from IPython.display import display

output_root = Path("outputs/governance")

def show_governance_summary(mode: str) -> None:
    """Read and display the current dataset summary for one run mode."""
    config_stem = Path(CONFIG_PATH).stem
    print(f"\n===== {mode} results =====")

    dataset_summary = output_root / mode / "dataset_summary.csv"
    fold_summary = output_root / mode / config_stem / "fold_summary.csv"
    sample_records = output_root / mode / config_stem / "sample_records.csv"

    if dataset_summary.exists():
        print("Reading:", dataset_summary)
        summary_df = pd.read_csv(dataset_summary)
        if "dataset_name" in summary_df.columns:
            dataset_row = summary_df[summary_df["dataset_name"].astype(str) == config_stem]
            if dataset_row.empty:
                available = ", ".join(summary_df["dataset_name"].astype(str).tolist())
                print(f"No row for {config_stem}; available datasets: {available}")
            else:
                display(dataset_row.reset_index(drop=True))
        else:
            print("dataset_summary.csv has no dataset_name column; cannot filter by dataset.")
    else:
        print("Missing dataset_summary.csv:", dataset_summary)

    if fold_summary.exists():
        print("Reading:", fold_summary)
        display(pd.read_csv(fold_summary))
    else:
        print("Missing fold_summary.csv:", fold_summary)

    if sample_records.exists():
        print("Sample records:", sample_records)
    else:
        print("Missing sample_records.csv:", sample_records)


## 运行完整 governance 实验


In [4]:
import subprocess

cmd = [
    sys.executable,
    "scripts/run_governance_experiments.py",
    "--config",
    CONFIG_PATH,
]

print("运行命令：", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)

print("标准输出：")
print(result.stdout)

print("错误输出：")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"实验运行失败，返回码：{result.returncode}")

show_governance_summary("full")


运行命令： d:\Anaconda3\python.exe scripts/run_governance_experiments.py --config configs/diabetic_bttwd.yaml
标准输出：
【INFO】【2026-05-18 19:37:48】【数据加载】文本表格 E:\yan\组\三支决策\机器学习\BT_TWD_v2\data\diabetic\diabetic_data.csv 已读取，样本数=101766，列数=50
【INFO】【2026-05-18 19:37:48】【数据加载】标签列 readmitted 已处理完成：dropna_target=False, 丢弃样本=0, 最终样本数=101766, 正类比例=11.16%
【INFO】【2026-05-18 19:37:48】【数据集信息】名称=diabetic，样本数=101766，目标列=readmitted，正类比例=11.16%
【INFO】【2026-05-18 19:37:48】【预处理】连续特征=8个，类别特征=35个
【INFO】【2026-05-18 19:37:50】【预处理】编码后维度=215
【INFO】【2026-05-18 19:37:51】【governance】diabetic_bttwd 第 1/5 折
【INFO】【2026-05-18 19:37:54】【桶树】已为样本生成桶ID，共 45 个组合
【INFO】【2026-05-18 19:37:54】【桶树】已为样本生成桶ID，共 42 个组合
【INFO】【2026-05-18 19:38:07】【governance】weak bucket resolver：strong=20，weak=38
【INFO】【2026-05-18 19:38:07】【桶树】已为样本生成桶ID，共 43 个组合
【INFO】【2026-05-18 19:38:50】【governance】diabetic_bttwd 第 2/5 折
【INFO】【2026-05-18 19:38:52】【桶树】已为样本生成桶ID，共 45 个组合
【INFO】【2026-05-18 19:38:52】【桶树】已为样本生成桶ID，共 45 个组合
【INFO】【2026-05-18 19:39:04】【gover

,dataset_name,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,closure_rate,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,diabetic_bttwd,0.408918,0.574977,0.247891,0.85336,0.394154,0.524234,0.10273,0.885905,1.0,...,0.0,1.0,0.0,1.0,0.0,0.272727,-0.272727,-1.0,0.0,12.0


Reading: outputs\governance\full\diabetic_bttwd\fold_summary.csv


,dataset_name,fold_id,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,diabetic_bttwd,1,0.411172,0.575489,0.248229,0.848826,0.401567,0.524653,0.103004,0.887049,...,0.0,1.0,0.0,0.0,0.0,0.000000,0.000000,-1.0,0.0,0.0
1,diabetic_bttwd,2,0.402987,0.581141,0.260367,0.855402,0.389672,0.527216,0.114760,0.884784,...,0.0,1.0,0.0,1.0,0.0,0.500000,-0.500000,-1.0,0.0,4.0
2,diabetic_bttwd,3,0.410406,0.573241,0.244444,0.852995,0.394708,0.521635,0.094054,0.885471,...,0.0,1.0,0.0,0.0,0.0,0.000000,0.000000,-1.0,0.0,0.0
3,diabetic_bttwd,4,0.404412,0.578367,0.255232,0.856630,0.388346,0.528403,0.117378,0.886208,...,0.0,1.0,0.0,1.0,0.0,0.666667,-0.666667,-1.0,0.0,8.0
4,diabetic_bttwd,5,0.415614,0.566650,0.231184,0.852946,0.396477,0.519261,0.084451,0.886012,...,0.0,1.0,0.0,0.0,0.0,0.000000,0.000000,-1.0,0.0,0.0


Sample records: outputs\governance\full\diabetic_bttwd\sample_records.csv


## 运行无 CP 消融


In [ ]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-cp-validation",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无 CP 消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_cp")
else:
    print("RUN_ABLATION=False，跳过无 CP 消融。")


运行命令： d:\Anaconda3\python.exe scripts/run_governance_experiments.py --config configs/diabetic_bttwd.yaml --disable-cp-validation


## 运行无渐进更新消融


In [ ]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-progressive-update",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无渐进更新消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_progressive")
else:
    print("RUN_ABLATION=False，跳过无渐进更新消融。")
